# Лабораторна робота №2 “Пошук асоціативних правил”

## Мета
Ознайомитися з принципами побудови асоціативних правил (association rule mining) на реальних даних користувацьких уподобань. Навчитись виконувати пошук частих наборів елементів (frequent itemsets) і формувати асоціативні правила за допомогою метрик support, confidence і lift на основі набору даних MovieLens Small. Розвинути навички попередньої обробки даних, перетворення даних у транзакційний формат і інтерпретації результатів.

## Завантаження та підготовка даних
1. Викачати архів ml-latest-small.zip і розпакувати.
2. Завантажити таблиці ratings.csv і movies.csv за допомогою pandas.
3. Виконати об’єднання таблиць за полем movieId.
4. Для подальшого аналізу вибрати лише користувацькі оцінки rating ≥ 4.0, вважаючи, що це “фільми що сподобались користувачам”.
5. Вивести розміри таблиць та кількість унікальних користувачів і фільмів.


## Перетворення даних у транзакційний формат
1. Створити матрицю (або DataFrame) у форматі:
 рядки — користувачі (userId),
 стовпці — фільми (title),
 значення — 1, якщо користувач поставив оцінку ≥ 4.0, і 0 інакше.
2. Вивести кілька перших рядків матриці для перевірки.


## Пошук частих наборів фільмів
1. Використати алгоритм Apriori для пошуку частих наборів елементів.
2. Вивести кількість знайдених частих наборів при різних порогах min_support (наприклад, 0.05, 0.1, 0.3).
3. Відсортувати результати за значенням support і вивести топ-10 найпопулярніших комбінацій фільмів.
4. Інтерпретувати, які фільми часто зустрічаються разом у вподобаннях користувачів.


## Побудова асоціативних правил
1. Побудувати асоціативні правила.
2. Додати обчислення метрик support, confidence, lift.
3. Відсортувати та вивести топ-10 правил за lift.
4. Інтерпретувати кілька правил.
5. Візуалізувати розподіл метрик (графіки support–confidence, lift–confidence тощо).


## Аналіз та висновки
1. Зробити висновки щодо того, як асоціативні правила можна застосувати у системах рекомендацій.
2. Оцінити вплив параметрів min_support і min_confidence на кількість і якість знайдених правил.


In [1]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

ratings = pd.read_csv("ratings.csv")
movies = pd.read_csv("movies.csv")

data = pd.merge(ratings, movies, on='movieId')
liked_movies = data[data['rating'] >= 4.0]

print(f"Розмір повної об’єднаної таблиці: {data.shape}")
print(f"Розмір таблиці з улюбленими фільмами: {liked_movies.shape}")

num_users = liked_movies['userId'].nunique()
num_movies = liked_movies['movieId'].nunique()

print(f"Кількість унікальних користувачів: {num_users}")
print(f"Кількість унікальних фільмів: {num_movies}")

Розмір повної об’єднаної таблиці: (100836, 6)
Розмір таблиці з улюбленими фільмами: (48580, 6)
Кількість унікальних користувачів: 609
Кількість унікальних фільмів: 6298


In [2]:
liked_movies = liked_movies.copy()
liked_movies['like'] = True

user_movie_matrix = liked_movies.pivot_table(
    index ='userId', 
    columns ='title', 
    values ='like', 
    aggfunc ='max',
    fill_value = False
)

print(user_movie_matrix.iloc[:5, :5])

title   '71 (2014)  'Hellboy': The Seeds of Creation (2004)  \
userId                                                        
1            False                                    False   
2            False                                    False   
3            False                                    False   
4            False                                    False   
5            False                                    False   

title   'Salem's Lot (2004)  'Til There Was You (1997)  'burbs, The (1989)  
userId                                                                      
1                     False                      False               False  
2                     False                      False               False  
3                     False                      False               False  
4                     False                      False               False  
5                     False                      False               False  


In [3]:
min_support_list = [0.05, 0.1, 0.3]

for support in min_support_list:
    frequent_itemsets = apriori(user_movie_matrix, min_support=support, use_colnames=True)
    print(f"min_support = {support}: {len(frequent_itemsets)}")

frequent_itemsets = apriori(user_movie_matrix, min_support=0.05, use_colnames=True)
frequent_itemsets = frequent_itemsets.sort_values(by='support', ascending=False)

print("Топ 10 комбінацій:")
print(frequent_itemsets.head(10))

min_support = 0.05: 33189
min_support = 0.1: 863
min_support = 0.3: 6
Топ 10 комбінацій:
       support                                           itemsets
267   0.449918      frozenset({Shawshank Redemption, The (1994)})
119   0.408867                   frozenset({Forrest Gump (1994)})
244   0.400657                   frozenset({Pulp Fiction (1994)})
273   0.369458      frozenset({Silence of the Lambs, The (1991)})
206   0.364532                    frozenset({Matrix, The (1999)})
299   0.330049  frozenset({Star Wars: Episode IV - A New Hope ...
115   0.293924                     frozenset({Fight Club (1999)})
260   0.287356               frozenset({Schindler's List (1993)})
2228  0.275862  frozenset({Shawshank Redemption, The (1994), F...
300   0.275862  frozenset({Star Wars: Episode V - The Empire S...


In [6]:
min_confidence = 0.3
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=min_confidence)

print("Асоціативні правила:")
print(f"Мінімальна довіра: {min_confidence}")
print(f"Кількість правил: {len(rules)}")

if len(rules) > 0:
    rules_display = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].copy()
    rules_display['правило'] = (
        rules_display['antecedents'].astype(str) + ' → ' +
        rules_display['consequents'].astype(str)
    )

    print("\nАналіз якості правил:")
    print(f"Середня довіра: {rules['confidence'].mean():.3f}")
    print(f"Середній lift: {rules['lift'].mean():.3f}")
    print(f"Кількість правил з lift > 1: {len(rules[rules['lift'] > 1])}")
else:
    print("Для заданого порогу довіри правил не знайдено.")
print("\nТоп правил:") 
rules_display.sort_values('confidence', ascending=False)

Асоціативні правила:
Мінімальна довіра: 0.3
Кількість правил: 360465

Аналіз якості правил:
Середня довіра: 0.578
Середній lift: 3.705
Кількість правил з lift > 1: 360464

Топ правил:


,antecedents,consequents,support,confidence,lift,правило
299438,"frozenset({Seven (a.k.a. Se7en) (1995), Lord o...",frozenset({Lord of the Rings: The Return of th...,0.050903,1.0,4.350000,"frozenset({'Seven (a.k.a. Se7en) (1995)', 'Lor..."
299332,"frozenset({Seven (a.k.a. Se7en) (1995), Lord o...",frozenset({Lord of the Rings: The Return of th...,0.050903,1.0,4.350000,"frozenset({'Seven (a.k.a. Se7en) (1995)', 'Lor..."
299364,"frozenset({Matrix, The (1999), Lord of the Rin...",frozenset({Lord of the Rings: The Return of th...,0.050903,1.0,4.350000,"frozenset({'Matrix, The (1999)', 'Lord of the ..."
299415,"frozenset({Shawshank Redemption, The (1994), L...",frozenset({Lord of the Rings: The Return of th...,0.050903,1.0,4.350000,"frozenset({'Shawshank Redemption, The (1994)',..."
342066,frozenset({Star Wars: Episode IV - A New Hope ...,frozenset({Pulp Fiction (1994)}),0.050903,1.0,2.495902,frozenset({'Star Wars: Episode IV - A New Hope...
...,...,...,...,...,...,...
32986,"frozenset({Shawshank Redemption, The (1994), S...","frozenset({Seven (a.k.a. Se7en) (1995), Pulp F...",0.068966,0.3,2.610000,"frozenset({'Shawshank Redemption, The (1994)',..."
32993,frozenset({Seven (a.k.a. Se7en) (1995)}),"frozenset({Shawshank Redemption, The (1994), P...",0.068966,0.3,2.573239,frozenset({'Seven (a.k.a. Se7en) (1995)'}) → f...
186901,frozenset({Alien (1979)}),"frozenset({Shawshank Redemption, The (1994), S...",0.054187,0.3,2.228049,frozenset({'Alien (1979)'}) → frozenset({'Shaw...
209718,"frozenset({Matrix, The (1999), Silence of the ...","frozenset({American History X (1998), Fight Cl...",0.054187,0.3,4.350000,"frozenset({'Matrix, The (1999)', 'Silence of t..."


# Висновок
У ході роботи було виконано пошук асоціативних правил для набору даних MovieLens Small. Було сформовано матрицю вподобань користувачів, знайдено часті набори фільмів та побудовано правила виду "якщо користувачу сподобався фільм A, то ймовірно сподобається і фільм B". Асоціативні правила в системах рекомендацій можна застосовувати для формування списку фільмів, які можна запропонувати користувачу після перегляду або високої оцінки іншого фільму. Аналіз support, confidence та lift показав, що зі зменшенням мінімальної підтримки кількість знайдених наборів і правил збільшується, але вони стають менш значущими.